In [1]:
!wget https://raw.githubusercontent.com/cbuntain/umd.inst414/refs/heads/main/Module07/prediction_model_01.csv
!wget https://raw.githubusercontent.com/cbuntain/umd.inst414/refs/heads/main/Module07/prediction_model_02.csv
!wget https://raw.githubusercontent.com/cbuntain/umd.inst414/refs/heads/main/Module07/prediction_model_03.csv

--2026-11-20 19:49:14--  https://raw.githubusercontent.com/cbuntain/umd.inst414/refs/heads/main/Module07/prediction_model_01.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 83501 (82K) [text/plain]
Saving to: ‘prediction_model_01.csv’

prediction_model_01 100%[===================>]  81.54K  --.-KB/s    in 0.01s   

2026-11-20 19:49:14 (5.65 MB/s) - ‘prediction_model_01.csv’ saved [83501/83501]

--2026-11-20 19:49:14--  https://raw.githubusercontent.com/cbuntain/umd.inst414/refs/heads/main/Module07/prediction_model_02.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request

In [2]:
import ast
import pandas as pd

dfs = {}
for i in range(1, 4):
    df = pd.read_csv(f'prediction_model_0{i}.csv')
    df['actual genres'] = df['actual genres'].apply(ast.literal_eval)
    dfs["Model %d" % i] = df

In [19]:
df["actual genres"][0], type(df["actual genres"])

(['Horror'], pandas.Series)

In [20]:
df["correct?"].value_counts()

correct?
0    946
1    939
Name: count, dtype: int64

In [21]:
for name, df in dfs.items():
    print(name, "acc:", sum(df["correct?"])/len(df))

Model 1 acc: 0.08594164456233422
Model 2 acc: 0.5517241379310345
Model 3 acc: 0.4981432360742706


In [22]:
df.head(3)

,imdb_id,predicted,actual genres,correct?
0,tt2390341,Drama,[Horror],0
1,tt3343136,Drama,"[Adventure, Fantasy, Horror]",0
2,tt6423362,Drama,"[Drama, Music, Romance]",1


In [24]:
def genre_metrics(df):
    genre_true_counts = {}
    genre_tp_counts = {}
    genre_fp_counts = {}

    for _, row in df.iterrows():
        true_genre_list = row["actual genres"]
        for genre in true_genre_list:
            genre_true_counts[genre] = genre_true_counts.get(genre, 0) + 1

        pred_genre = row["predicted"]
        if pred_genre in true_genre_list:
            genre_tp_counts[pred_genre] = genre_tp_counts.get(pred_genre, 0) + 1
        else:
            genre_fp_counts[pred_genre] = genre_fp_counts.get(pred_genre, 0) + 1

    metric_rows = []
    for genre in sorted(set(genre_true_counts) | set(genre_tp_counts) | set(genre_fp_counts)):
        tp = genre_tp_counts.get(genre, 0)
        fp = genre_fp_counts.get(genre, 0)
        fn = genre_true_counts.get(genre, 0) - tp

        precision = tp / (tp + fp) if tp > 0 else 0.0
        recall = tp / (tp +fn) if tp > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if tp > 0 else 0

        metric_rows.append({
            "genre": genre,
            "times_predicted": tp + fp,
            "times_actual": tp + fn,
            "precision": precision,
            "recall": recall,
            "f1": f1
        })
    return pd.DataFrame(metric_rows).sort_values(by=["times_predicted", "genre"], ascending=[False, True], ignore_index = True)

metrics = {name: genre_metrics(df) for name, df in dfs.items()}

for name in metrics:
    print(name)
    display(metrics[name].head(10).round(3))

Model 1


,genre,times_predicted,times_actual,precision,recall,f1
0,Documentary,102,63,0.039,0.063,0.048
1,Action,94,367,0.245,0.063,0.100
2,Reality-TV,90,0,0.000,0.000,0.000
3,Short,89,0,0.000,0.000,0.000
4,Adventure,86,171,0.081,0.041,0.054
5,News,85,3,0.000,0.000,0.000
6,Music,84,42,0.000,0.000,0.000
7,Horror,83,316,0.145,0.038,0.060
8,Drama,82,932,0.500,0.044,0.081
9,Western,82,28,0.024,0.071,0.036


Model 2


,genre,times_predicted,times_actual,precision,recall,f1
0,Drama,1013,943,0.622,0.668,0.644
1,Comedy,311,586,0.527,0.280,0.366
2,Action,175,351,0.531,0.265,0.354
3,Horror,162,305,0.599,0.318,0.415
4,Thriller,101,355,0.238,0.068,0.105
5,Documentary,70,56,0.086,0.107,0.095
6,Animation,13,63,0.923,0.190,0.316
7,Crime,11,273,0.273,0.011,0.021
8,Adventure,9,165,0.333,0.018,0.034
9,Biography,6,91,0.500,0.033,0.062


Model 3


,genre,times_predicted,times_actual,precision,recall,f1
0,Drama,1885,939,0.498,1.0,0.665
1,,0,12,0.000,0.0,0.000
2,Action,0,345,0.000,0.0,0.000
3,Adventure,0,163,0.000,0.0,0.000
4,Animation,0,57,0.000,0.0,0.000
5,Biography,0,74,0.000,0.0,0.000
6,Comedy,0,590,0.000,0.0,0.000
7,Crime,0,260,0.000,0.0,0.000
8,Documentary,0,60,0.000,0.0,0.000
9,Family,0,91,0.000,0.0,0.000


In [25]:
summary_df = pd.DataFrame(
    {"model": name, "accuracy": dfs[name]["correct?"].mean(), "macro f1": metrics[name]["f1"].mean()}
    for name in dfs
)

summary_df

,model,accuracy,macro f1
0,Model 1,0.085942,0.033588
1,Model 2,0.551724,0.113996
2,Model 3,0.498143,0.028914


In [26]:
# acc 2 > 3 > 1
# f1: 2 > 1 > 3

In [27]:
#Model 2 Animation has high precision and low recall
#Model 3 Drama has high recall and low precision